<cell_type>markdown</cell_type># 高级推理引擎技术教程 (Advanced Inference Techniques Tutorial)

> **前置知识**: ONNX Runtime、TensorRT、vLLM 基础
>
> **学习目标**: 掌握生产环境中的高级推理技术

---

## 为什么需要高级推理技术？

```
┌─────────────────────────────────────────────────────────────┐
│                    生产环境挑战                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  基础推理                        生产环境需求               │
│  ┌─────────────────┐            ┌─────────────────┐        │
│  │ 单请求处理      │            │ 高并发处理      │        │
│  │ 同步阻塞        │     →      │ 异步非阻塞      │        │
│  │ 固定批次        │            │ 动态批处理      │        │
│  │ 无监控          │            │ 全面监控        │        │
│  │ 单点故障        │            │ 高可用降级      │        │
│  └─────────────────┘            └─────────────────┘        │
│                                                             │
│  本教程涵盖的高级技术:                                      │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  1. 异步推理管理器 - 提高并发处理能力               │   │
│  │  2. 动态批处理器 - 自动收集请求批量处理             │   │
│  │  3. 性能分析器 - 延迟分布和吞吐量统计               │   │
│  │  4. 健康监控 - 成功率、延迟指标监控                 │   │
│  │  5. A/B 测试框架 - 模型版本对比                     │   │
│  │  6. 优雅降级 - 故障时自动切换备用模型               │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

## 本教程内容

1. **ONNX Runtime 高级优化** - 会话配置和性能调优
2. **异步推理管理器** - 并发请求处理
3. **动态批处理器** - 自动批量处理
4. **性能分析器** - 延迟和吞吐量分析
5. **健康监控** - 生产环境监控
6. **A/B 测试框架** - 模型对比测试
7. **优雅降级服务** - 高可用保障

In [ ]:
# ============================================================
# 环境准备
# ============================================================
import numpy as np
import time
import threading
from typing import List, Dict, Any
from concurrent.futures import ThreadPoolExecutor
from queue import Queue
from collections import deque
from dataclasses import dataclass, field
import random

# 设置随机种子
np.random.seed(42)

# 检查可用的推理引擎
print("=" * 60)
print("环境准备完成")
print("=" * 60)

try:
    import onnxruntime as ort
    ORT_AVAILABLE = True
    print(f"\n✓ ONNX Runtime 版本: {ort.__version__}")
    print(f"  可用 Providers: {ort.get_available_providers()}")
except ImportError:
    ORT_AVAILABLE = False
    print("\n✗ ONNX Runtime 未安装")

try:
    import torch
    TORCH_AVAILABLE = True
    print(f"✓ PyTorch 版本: {torch.__version__}")
except ImportError:
    TORCH_AVAILABLE = False
    print("✗ PyTorch 未安装")

<cell_type>markdown</cell_type>## 1. ONNX Runtime 高级优化

**核心概念**: 通过会话配置和性能调优实现最佳推理性能

```
┌─────────────────────────────────────────────────────────────┐
│                    ONNX Runtime 优化配置                     │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  SessionOptions 关键配置:                                   │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  图优化级别:                                        │   │
│  │  - ORT_DISABLE_ALL: 禁用优化 (调试用)              │   │
│  │  - ORT_ENABLE_BASIC: 基本优化                      │   │
│  │  - ORT_ENABLE_EXTENDED: 扩展优化                   │   │
│  │  - ORT_ENABLE_ALL: 全部优化 (推荐)                 │   │
│  │                                                     │   │
│  │  线程配置:                                          │   │
│  │  - intra_op_num_threads: 算子内并行                │   │
│  │  - inter_op_num_threads: 算子间并行                │   │
│  │                                                     │   │
│  │  内存优化:                                          │   │
│  │  - enable_mem_pattern: 内存模式优化                │   │
│  │  - enable_mem_reuse: 内存复用                      │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  Execution Provider 选择:                                   │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  GPU 可用: ['CUDAExecutionProvider', 'CPU...']     │   │
│  │  仅 CPU:   ['CPUExecutionProvider']                │   │
│  │                                                     │   │
│  │  Provider 按优先级排序，自动选择最优               │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 优化的 ONNX Runtime 会话
# ============================================================
print("=" * 60)
print("优化的 ONNX Runtime 会话")
print("=" * 60)

if ORT_AVAILABLE:
    class OptimizedONNXSession:
        """
        优化的 ONNX Runtime 会话
        
        封装了最佳实践配置，包括:
        - 图优化级别
        - 线程配置
        - 内存优化
        - Provider 选择
        """
        def __init__(self, model_path, use_gpu=False):
            # 创建会话选项
            self.sess_options = ort.SessionOptions()
            
            # 图优化: 启用所有优化
            self.sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
            
            # 线程配置
            self.sess_options.intra_op_num_threads = 4  # 算子内并行
            self.sess_options.inter_op_num_threads = 2  # 算子间并行
            
            # 内存优化
            self.sess_options.enable_mem_pattern = True   # 内存模式优化
            self.sess_options.enable_mem_reuse = True     # 内存复用
            
            # 选择 Execution Provider
            if use_gpu and 'CUDAExecutionProvider' in ort.get_available_providers():
                providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
            else:
                providers = ['CPUExecutionProvider']
            
            # 创建会话
            self.session = ort.InferenceSession(
                model_path, 
                self.sess_options, 
                providers=providers
            )
            self.input_name = self.session.get_inputs()[0].name
            self.providers = self.session.get_providers()
        
        def infer(self, input_data):
            """执行推理"""
            return self.session.run(None, {self.input_name: input_data})[0]
        
        def benchmark(self, input_data, warmup=10, iterations=100):
            """
            性能基准测试
            
            参数:
                input_data: 输入数据
                warmup: 预热次数
                iterations: 测试迭代次数
                
            返回:
                dict: 性能统计 (均值、标准差、百分位数)
            """
            # 预热
            for _ in range(warmup):
                self.infer(input_data)
            
            # 测量
            latencies = []
            for _ in range(iterations):
                start = time.perf_counter()
                self.infer(input_data)
                latencies.append((time.perf_counter() - start) * 1000)
            
            return {
                'mean_ms': np.mean(latencies),
                'std_ms': np.std(latencies),
                'p50_ms': np.percentile(latencies, 50),
                'p95_ms': np.percentile(latencies, 95),
                'p99_ms': np.percentile(latencies, 99),
                'throughput': 1000 / np.mean(latencies)
            }
    
    print("\n✓ OptimizedONNXSession 类定义完成")
    print("\n使用示例:")
    print("  session = OptimizedONNXSession('model.onnx', use_gpu=True)")
    print("  output = session.infer(input_data)")
    print("  stats = session.benchmark(input_data)")
else:
    print("\n跳过 (ONNX Runtime 未安装)")

<cell_type>markdown</cell_type>## 2. 异步推理管理器

**核心概念**: 使用线程池实现并发推理，提高吞吐量

```
┌─────────────────────────────────────────────────────────────┐
│                    异步推理架构                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  同步推理 (阻塞):                                           │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  请求1 → [处理] → 响应1                             │   │
│  │                    请求2 → [处理] → 响应2           │   │
│  │                                      请求3 → ...    │   │
│  │  ↑ 必须等待前一个完成                               │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  异步推理 (并发):                                           │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  请求1 → [Worker 1] → 响应1                        │   │
│  │  请求2 → [Worker 2] → 响应2                        │   │
│  │  请求3 → [Worker 3] → 响应3                        │   │
│  │  请求4 → [Worker 4] → 响应4                        │   │
│  │  ↑ 多个请求并行处理                                 │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  实现方式:                                                  │
│  - ThreadPoolExecutor: 线程池管理                          │
│  - Future: 异步结果获取                                    │
│  - Queue: 请求队列管理                                     │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 异步推理管理器
# ============================================================
print("=" * 60)
print("异步推理管理器")
print("=" * 60)

class AsyncInferenceManager:
    """
    异步推理管理器
    
    使用线程池实现并发推理，提高吞吐量
    
    特点:
    - 非阻塞提交请求
    - 线程池管理并发
    - 支持结果异步获取
    """
    def __init__(self, model_fn, max_workers=4):
        """
        参数:
            model_fn: 推理函数
            max_workers: 最大工作线程数
        """
        self.model_fn = model_fn
        self.executor = ThreadPoolExecutor(max_workers=max_workers)
        self.results = {}
        self.lock = threading.Lock()
        self.request_id = 0
    
    def submit(self, input_data):
        """
        提交推理请求 (非阻塞)
        
        返回:
            req_id: 请求 ID
            future: Future 对象，用于获取结果
        """
        with self.lock:
            req_id = self.request_id
            self.request_id += 1
        
        future = self.executor.submit(self._process, req_id, input_data)
        return req_id, future
    
    def _process(self, req_id, input_data):
        """处理单个请求"""
        result = self.model_fn(input_data)
        with self.lock:
            self.results[req_id] = result
        return result
    
    def get_result(self, req_id):
        """获取已完成的结果"""
        with self.lock:
            return self.results.pop(req_id, None)
    
    def shutdown(self):
        """关闭线程池"""
        self.executor.shutdown(wait=True)


# ============================================================
# 示例使用
# ============================================================
def dummy_model(x):
    """模拟推理函数"""
    time.sleep(0.01)  # 模拟推理延迟
    return x * 2

manager = AsyncInferenceManager(dummy_model, max_workers=4)

# 提交多个请求 (非阻塞)
print("\n提交 10 个异步请求...")
futures = []
start = time.perf_counter()

for i in range(10):
    req_id, future = manager.submit(np.array([i]))
    futures.append((req_id, future))

# 等待所有结果
print("\n等待结果:")
for req_id, future in futures:
    result = future.result()
    print(f"  请求 {req_id}: 输入={req_id}, 输出={result[0]}")

elapsed = time.perf_counter() - start
print(f"\n总耗时: {elapsed*1000:.1f} ms")
print(f"理论串行耗时: {10 * 10:.1f} ms")
print(f"加速比: {(10 * 10) / (elapsed * 1000):.1f}x")

manager.shutdown()

<cell_type>markdown</cell_type>## 3. 动态批处理器

**核心概念**: 自动收集请求并批量处理，提高 GPU 利用率

```
┌─────────────────────────────────────────────────────────────┐
│                    动态批处理原理                            │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  单请求处理 (低效):                                         │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  请求1 → [GPU 推理] → 响应1                        │   │
│  │  请求2 → [GPU 推理] → 响应2                        │   │
│  │  请求3 → [GPU 推理] → 响应3                        │   │
│  │  ↑ GPU 利用率低，大量时间在等待                    │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  动态批处理 (高效):                                         │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  请求1 ─┐                                          │   │
│  │  请求2 ─┼─→ [收集] → [批量 GPU 推理] → 分发响应   │   │
│  │  请求3 ─┘                                          │   │
│  │  ↑ GPU 一次处理多个请求，利用率高                  │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  触发条件:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  1. 达到最大批次大小 (max_batch_size)              │   │
│  │  2. 等待超时 (max_wait_ms)                         │   │
│  │  满足任一条件即触发批量处理                        │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 动态批处理器
# ============================================================
print("=" * 60)
print("动态批处理器")
print("=" * 60)

class DynamicBatcher:
    """
    动态批处理器
    
    自动收集请求并批量处理，提高 GPU 利用率
    
    触发条件:
    - 达到最大批次大小 (max_batch_size)
    - 等待超时 (max_wait_ms)
    """
    def __init__(self, model_fn, max_batch_size=32, max_wait_ms=10):
        """
        参数:
            model_fn: 批量推理函数 (接受批量输入)
            max_batch_size: 最大批次大小
            max_wait_ms: 最大等待时间 (毫秒)
        """
        self.model_fn = model_fn
        self.max_batch_size = max_batch_size
        self.max_wait_ms = max_wait_ms
        self.pending = []
        self.lock = threading.Lock()
        self.batch_count = 0
        self.total_requests = 0
    
    def add_request(self, input_data):
        """
        添加请求
        
        当达到批次大小或超时时，触发批量处理
        """
        with self.lock:
            self.pending.append(input_data)
            self.total_requests += 1
            
            # 条件 1: 达到最大批次大小
            if len(self.pending) >= self.max_batch_size:
                return self._process_batch()
        
        # 等待更多请求或超时
        time.sleep(self.max_wait_ms / 1000)
        
        # 条件 2: 超时后处理剩余请求
        with self.lock:
            if self.pending:
                return self._process_batch()
        return None
    
    def _process_batch(self):
        """处理一个批次"""
        batch = self.pending[:self.max_batch_size]
        self.pending = self.pending[self.max_batch_size:]
        self.batch_count += 1
        
        # 批量推理
        batch_input = np.stack(batch)
        results = self.model_fn(batch_input)
        return results
    
    def get_stats(self):
        """获取统计信息"""
        return {
            'total_requests': self.total_requests,
            'batch_count': self.batch_count,
            'avg_batch_size': self.total_requests / max(1, self.batch_count)
        }


# ============================================================
# 示例使用
# ============================================================
def batch_model(x):
    """模拟批量推理"""
    return x * 2

batcher = DynamicBatcher(batch_model, max_batch_size=4, max_wait_ms=5)

# 模拟请求
print("\n模拟 8 个请求 (批次大小=4):")
for i in range(8):
    result = batcher.add_request(np.array([i, i+1]))
    if result is not None:
        print(f"  批次 {batcher.batch_count}: 输入形状={result.shape}")

stats = batcher.get_stats()
print(f"\n统计信息:")
print(f"  总请求数: {stats['total_requests']}")
print(f"  批次数: {stats['batch_count']}")
print(f"  平均批次大小: {stats['avg_batch_size']:.1f}")

<cell_type>markdown</cell_type>## 4. 性能分析器

**核心概念**: 收集延迟分布和吞吐量统计，指导优化决策

```
┌─────────────────────────────────────────────────────────────┐
│                    性能指标说明                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  延迟指标:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  Mean: 平均延迟 - 整体性能指标                      │   │
│  │  P50:  中位数延迟 - 典型用户体验                    │   │
│  │  P95:  95% 请求的延迟 - 大部分用户体验              │   │
│  │  P99:  99% 请求的延迟 - 尾延迟，影响 SLA            │   │
│  │  Max:  最大延迟 - 最坏情况                          │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  延迟分布示例:                                              │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  频率                                               │   │
│  │   ↑                                                 │   │
│  │   │    ████                                         │   │
│  │   │   ██████                                        │   │
│  │   │  ████████                                       │   │
│  │   │ ██████████ █                                    │   │
│  │   │████████████████  █  █                           │   │
│  │   └──────────────────────────────→ 延迟             │   │
│  │        P50   P95  P99  Max                          │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  吞吐量: requests/sec 或 samples/sec                        │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 性能分析器
# ============================================================
print("=" * 60)
print("性能分析器")
print("=" * 60)

class InferenceProfiler:
    """
    推理性能分析器
    
    收集延迟分布和吞吐量统计，支持多模型对比
    """
    def __init__(self):
        self.records = []
    
    def profile(self, model_fn, input_data, name='model', warmup=5, iterations=50):
        """
        分析模型性能
        
        参数:
            model_fn: 推理函数
            input_data: 输入数据
            name: 模型名称
            warmup: 预热次数
            iterations: 测试迭代次数
            
        返回:
            dict: 性能统计
        """
        # 预热
        for _ in range(warmup):
            model_fn(input_data)
        
        # 测量
        latencies = []
        for _ in range(iterations):
            start = time.perf_counter()
            output = model_fn(input_data)
            latencies.append((time.perf_counter() - start) * 1000)
        
        record = {
            'name': name,
            'mean_ms': np.mean(latencies),
            'std_ms': np.std(latencies),
            'min_ms': np.min(latencies),
            'max_ms': np.max(latencies),
            'p50_ms': np.percentile(latencies, 50),
            'p95_ms': np.percentile(latencies, 95),
            'p99_ms': np.percentile(latencies, 99),
            'throughput': 1000 / np.mean(latencies),
        }
        self.records.append(record)
        return record
    
    def compare(self):
        """对比所有已分析的模型"""
        print('\n=== 性能对比 ===')
        print(f'{"模型":<25} {"Mean(ms)":<12} {"P99(ms)":<12} {"吞吐量(/s)":<12}')
        print('-' * 61)
        for r in self.records:
            print(f'{r["name"]:<25} {r["mean_ms"]:<12.3f} {r["p99_ms"]:<12.3f} {r["throughput"]:<12.1f}')
    
    def clear(self):
        """清空记录"""
        self.records = []


# ============================================================
# 示例使用
# ============================================================
profiler = InferenceProfiler()

# 模拟不同版本的模型
def model_v1(x): 
    time.sleep(0.001)
    return x

def model_v2(x): 
    time.sleep(0.0008)  # 优化版本，更快
    return x

data = np.random.randn(1, 64).astype(np.float32)

print("\n分析模型性能...")
profiler.profile(model_v1, data, 'Model V1 (原始)')
profiler.profile(model_v2, data, 'Model V2 (优化)')
profiler.compare()

<cell_type>markdown</cell_type>## 5. 健康监控

**核心概念**: 生产环境需要实时监控推理服务的健康状态

```
┌─────────────────────────────────────────────────────────────┐
│                    健康监控指标                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  核心指标:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  成功率 = 成功请求数 / 总请求数                     │   │
│  │  平均延迟 = sum(延迟) / 请求数                      │   │
│  │  错误率 = 失败请求数 / 总请求数                     │   │
│  │  QPS = 请求数 / 时间窗口                            │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  健康检查端点:                                              │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  GET /health                                        │   │
│  │  {                                                  │   │
│  │    "healthy": true,                                 │   │
│  │    "success_rate": "99.5%",                        │   │
│  │    "avg_latency_ms": "15.2",                       │   │
│  │    "total_requests": 10000                         │   │
│  │  }                                                  │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  告警阈值:                                                  │
│  - 成功率 < 99% → 警告                                     │
│  - 成功率 < 95% → 严重                                     │
│  - P99 延迟 > SLA → 告警                                   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 带监控的推理引擎
# ============================================================
print("=" * 60)
print("带监控的推理引擎")
print("=" * 60)

@dataclass
class InferenceMetrics:
    """
    推理指标数据类
    
    使用 deque 存储最近的延迟数据，避免内存无限增长
    """
    total_requests: int = 0
    successful_requests: int = 0
    failed_requests: int = 0
    latencies: deque = field(default_factory=lambda: deque(maxlen=1000))
    
    @property
    def success_rate(self):
        """成功率"""
        if self.total_requests == 0:
            return 0.0
        return self.successful_requests / self.total_requests
    
    @property
    def avg_latency_ms(self):
        """平均延迟"""
        if not self.latencies:
            return 0.0
        return np.mean(list(self.latencies))


class MonitoredInferenceEngine:
    """
    带监控的推理引擎
    
    自动收集推理指标，支持健康检查
    """
    def __init__(self, model_fn):
        self.model_fn = model_fn
        self.metrics = InferenceMetrics()
        self.is_healthy = True
    
    def infer(self, input_data):
        """执行推理并记录指标"""
        self.metrics.total_requests += 1
        start = time.perf_counter()
        
        try:
            result = self.model_fn(input_data)
            latency = (time.perf_counter() - start) * 1000
            self.metrics.latencies.append(latency)
            self.metrics.successful_requests += 1
            return result
        except Exception as e:
            self.metrics.failed_requests += 1
            raise
    
    def health_check(self):
        """健康检查端点"""
        return {
            'healthy': self.is_healthy,
            'success_rate': f'{self.metrics.success_rate:.2%}',
            'avg_latency_ms': f'{self.metrics.avg_latency_ms:.2f}',
            'total_requests': self.metrics.total_requests,
            'failed_requests': self.metrics.failed_requests,
        }


# ============================================================
# 示例使用
# ============================================================
engine = MonitoredInferenceEngine(lambda x: x * 2)

# 模拟 100 个请求
print("\n模拟 100 个请求...")
for i in range(100):
    engine.infer(np.array([i]))

health = engine.health_check()
print(f"\n健康检查结果:")
for key, value in health.items():
    print(f"  {key}: {value}")

<cell_type>markdown</cell_type>## 6. A/B 测试框架

**核心概念**: 对比不同模型版本的性能，指导模型迭代决策

```
┌─────────────────────────────────────────────────────────────┐
│                    A/B 测试架构                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  流量分配:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │                                                     │   │
│  │  请求 ──→ [流量分配器] ──┬──→ Model A (50%)        │   │
│  │                         │                          │   │
│  │                         └──→ Model B (50%)        │   │
│  │                                                     │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  收集指标:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  Model A:                                           │   │
│  │  - 请求数: 1000                                    │   │
│  │  - 平均延迟: 15.2 ms                               │   │
│  │  - P99 延迟: 25.1 ms                               │   │
│  │                                                     │   │
│  │  Model B:                                           │   │
│  │  - 请求数: 1000                                    │   │
│  │  - 平均延迟: 12.8 ms  ← 更快                       │   │
│  │  - P99 延迟: 20.3 ms                               │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  决策: Model B 性能更好，可以全量切换                       │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# A/B 测试推理引擎
# ============================================================
print("=" * 60)
print("A/B 测试推理引擎")
print("=" * 60)

class ABTestingEngine:
    """
    A/B 测试推理引擎
    
    按比例分配流量到不同模型，收集性能指标进行对比
    """
    def __init__(self, engine_a, engine_b, traffic_ratio=0.5):
        """
        参数:
            engine_a: 模型 A (对照组)
            engine_b: 模型 B (实验组)
            traffic_ratio: 分配给模型 A 的流量比例
        """
        self.engine_a = engine_a
        self.engine_b = engine_b
        self.ratio = traffic_ratio
        self.metrics_a = {'latencies': [], 'count': 0}
        self.metrics_b = {'latencies': [], 'count': 0}
    
    def infer(self, input_data):
        """执行推理，按比例分配到不同模型"""
        if random.random() < self.ratio:
            engine, metrics = self.engine_a, self.metrics_a
        else:
            engine, metrics = self.engine_b, self.metrics_b
        
        start = time.perf_counter()
        result = engine(input_data)
        latency = (time.perf_counter() - start) * 1000
        
        metrics['latencies'].append(latency)
        metrics['count'] += 1
        return result
    
    def get_statistics(self):
        """获取 A/B 测试统计结果"""
        def calc_stats(metrics):
            if not metrics['latencies']:
                return {'count': 0, 'avg_ms': 0, 'p99_ms': 0}
            return {
                'count': metrics['count'],
                'avg_ms': np.mean(metrics['latencies']),
                'p99_ms': np.percentile(metrics['latencies'], 99),
            }
        return {
            'engine_a': calc_stats(self.metrics_a),
            'engine_b': calc_stats(self.metrics_b),
        }


# ============================================================
# 示例使用
# ============================================================
def engine_a(x): 
    time.sleep(0.001)
    return x

def engine_b(x): 
    time.sleep(0.0008)  # 优化版本，更快
    return x

ab_test = ABTestingEngine(engine_a, engine_b, traffic_ratio=0.5)

# 模拟 200 个请求
print("\n模拟 200 个请求 (50/50 流量分配)...")
for _ in range(200):
    ab_test.infer(np.array([1.0]))

stats = ab_test.get_statistics()
print(f"\nA/B 测试结果:")
print(f"  Model A (对照组): 请求数={stats['engine_a']['count']}, 平均延迟={stats['engine_a']['avg_ms']:.2f}ms")
print(f"  Model B (实验组): 请求数={stats['engine_b']['count']}, 平均延迟={stats['engine_b']['avg_ms']:.2f}ms")

if stats['engine_b']['avg_ms'] < stats['engine_a']['avg_ms']:
    improvement = (stats['engine_a']['avg_ms'] - stats['engine_b']['avg_ms']) / stats['engine_a']['avg_ms'] * 100
    print(f"\n结论: Model B 比 Model A 快 {improvement:.1f}%，建议全量切换")

<cell_type>markdown</cell_type>## 7. 优雅降级服务

**核心概念**: 当主模型出错时自动切换到备用模型，保证服务可用性

```
┌─────────────────────────────────────────────────────────────┐
│                    优雅降级架构                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  正常情况:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  请求 → [主模型] → 响应                             │   │
│  │         (高精度)                                    │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  主模型故障:                                                │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  请求 → [主模型] ✗ 失败                             │   │
│  │              ↓                                      │   │
│  │         [错误计数 +1]                               │   │
│  │              ↓                                      │   │
│  │         达到阈值?                                   │   │
│  │         ├── 否 → 重试主模型                        │   │
│  │         └── 是 → 切换到备用模型                    │   │
│  │                   ↓                                 │   │
│  │              [备用模型] → 响应 (降级服务)           │   │
│  │              (较低精度但可用)                       │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  降级策略:                                                  │
│  - 连续错误阈值: 3-5 次连续失败后切换                      │
│  - 备用模型: 更小/更稳定的模型                             │
│  - 自动恢复: 定期尝试恢复主模型                            │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 优雅降级服务
# ============================================================
print("=" * 60)
print("优雅降级服务")
print("=" * 60)

class ResilientInferenceService:
    """
    支持优雅降级的推理服务
    
    当主模型连续失败达到阈值时，自动切换到备用模型
    """
    def __init__(self, primary_engine, fallback_engine, error_threshold=3):
        """
        参数:
            primary_engine: 主模型 (高精度)
            fallback_engine: 备用模型 (高可用)
            error_threshold: 连续错误阈值，达到后切换
        """
        self.primary = primary_engine
        self.fallback = fallback_engine
        self.error_threshold = error_threshold
        self.consecutive_errors = 0
        self.use_fallback = False
        self.fallback_count = 0
        self.primary_count = 0
    
    def infer(self, input_data):
        """执行推理，自动处理故障切换"""
        if self.use_fallback:
            self.fallback_count += 1
            return self.fallback(input_data)
        
        try:
            result = self.primary(input_data)
            self.consecutive_errors = 0  # 成功后重置错误计数
            self.primary_count += 1
            return result
        except Exception as e:
            self.consecutive_errors += 1
            if self.consecutive_errors >= self.error_threshold:
                print(f"  ⚠ 主模型连续失败 {self.consecutive_errors} 次，切换到备用模型")
                self.use_fallback = True
            self.fallback_count += 1
            return self.fallback(input_data)
    
    def reset(self):
        """重置状态，尝试恢复主模型"""
        self.use_fallback = False
        self.consecutive_errors = 0
        print("  ✓ 已重置，尝试恢复主模型")
    
    def get_stats(self):
        """获取统计信息"""
        return {
            'primary_count': self.primary_count,
            'fallback_count': self.fallback_count,
            'use_fallback': self.use_fallback,
        }


# ============================================================
# 示例使用
# ============================================================
call_count = [0]

def unreliable_primary(x):
    """模拟不稳定的主模型 (每3次失败一次)"""
    call_count[0] += 1
    if call_count[0] % 3 == 0:
        raise RuntimeError('Primary model failed')
    return x * 2

def reliable_fallback(x):
    """可靠的备用模型 (精度略低)"""
    return x * 1.5

service = ResilientInferenceService(
    unreliable_primary, 
    reliable_fallback, 
    error_threshold=3
)

print("\n模拟 15 个请求 (主模型每3次失败一次):")
for i in range(15):
    result = service.infer(np.array([i + 1]))
    status = "降级" if service.use_fallback else "正常"
    print(f"  请求 {i+1}: 结果={result[0]:.1f}, 状态={status}")

stats = service.get_stats()
print(f"\n统计信息:")
print(f"  主模型处理: {stats['primary_count']} 次")
print(f"  备用模型处理: {stats['fallback_count']} 次")
print(f"  当前状态: {'降级模式' if stats['use_fallback'] else '正常模式'}")

## 总结

本教程介绍了生产环境中的高级推理技术：

### 核心知识点

| 技术 | 用途 | 关键实现 |
|:-----|:-----|:---------|
| 异步推理 | 提高并发处理能力 | ThreadPoolExecutor + Future |
| 动态批处理 | 提高 GPU 利用率 | 请求收集 + 批量处理 |
| 性能分析 | 指导优化决策 | 延迟分布 (P50/P95/P99) |
| 健康监控 | 生产环境运维 | 成功率/延迟/错误率 |
| A/B 测试 | 模型版本对比 | 流量分配 + 指标收集 |
| 优雅降级 | 高可用保障 | 错误阈值 + 自动切换 |

### 生产环境检查清单

```
部署前检查:
✓ 配置 ONNX Runtime 优化选项 (图优化、线程数)
✓ 实现异步推理支持高并发
✓ 配置动态批处理提高吞吐量
✓ 部署性能监控和告警
✓ 准备备用模型实现优雅降级
✓ 设置 A/B 测试框架支持模型迭代

性能指标:
✓ P99 延迟 < SLA 要求
✓ 成功率 > 99%
✓ 吞吐量满足业务需求
✓ GPU 利用率 > 80%
```

### API 速查

```python
# 异步推理
manager = AsyncInferenceManager(model_fn, max_workers=4)
req_id, future = manager.submit(input_data)
result = future.result()

# 动态批处理
batcher = DynamicBatcher(model_fn, max_batch_size=32, max_wait_ms=10)
result = batcher.add_request(input_data)

# 性能分析
profiler = InferenceProfiler()
stats = profiler.profile(model_fn, input_data, name='model_v1')
profiler.compare()

# 健康监控
engine = MonitoredInferenceEngine(model_fn)
health = engine.health_check()

# A/B 测试
ab_test = ABTestingEngine(engine_a, engine_b, traffic_ratio=0.5)
stats = ab_test.get_statistics()

# 优雅降级
service = ResilientInferenceService(primary, fallback, error_threshold=3)
result = service.infer(input_data)
```

### 下一步学习

- **03-serving-systems**: 模型服务系统 (FastAPI、Triton、负载均衡)